# Linear Regression

In [ ]:
library(tidyverse)
options(repr.matrix.max.rows = 20)

## Review: Watch Out For

In [ ]:
hybrid <- read_csv("data/hybrid.csv", show_col_types = FALSE)

ggplot(hybrid, aes(x = acceleration, y = msrp)) +
  geom_point() +
  theme_minimal(base_size = 16)

In [ ]:
cor(hybrid$acceleration, hybrid$msrp)

### Switching Axes

In [ ]:
ggplot(hybrid, aes(x = msrp, y = acceleration)) +
  geom_point() +
  theme_minimal(base_size = 16)

In [ ]:
cor(hybrid$msrp, hybrid$acceleration)

### Nonlinearity

In [ ]:
nonlinear <- tibble(
  x = seq(-4, 4, by = 0.5),
  y = x^2
)

ggplot(nonlinear, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size = 16)

In [ ]:
cor(nonlinear$x, nonlinear$y)

### Outliers

In [ ]:
line <- tibble(x = 1:4, y = 1:4)

ggplot(line, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size = 16)

In [ ]:
cor(line$x, line$y)

In [ ]:
outlier <- tibble(x = 1:5, y = c(1, 2, 3, 4, 0))

ggplot(outlier, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal(base_size = 16)

In [ ]:
cor(outlier$x, outlier$y)

### Ecological Correlations

In [ ]:
sat2014 <- read_csv("data/sat2014.csv", show_col_types = FALSE) |>
  arrange(State)

sat2014

In [ ]:
ggplot(sat2014, aes(x = `Critical Reading`, y = Math)) +
  geom_point() +
  theme_minimal(base_size = 16)

In [ ]:
cor(sat2014$`Critical Reading`, sat2014$Math)

Who took the test?

In [ ]:
sat2014 <- sat2014 |>
  mutate(rate_code = case_when(
    `Participation Rate` <= 25 ~ "low",
    `Participation Rate` <= 75 ~ "medium",
    TRUE ~ "high"
  ))

sat2014

In [ ]:
ggplot(sat2014, aes(x = `Critical Reading`, y = Math, color = rate_code)) +
  geom_point(size = 2) +
  scale_color_manual(values = c(low = "goldenrod", medium = "dodgerblue3", high = "red"),
                     breaks = c("low", "medium", "high")) +
  theme_minimal(base_size = 16)

In [ ]:
sat2014 |>
  filter(rate_code == "low")

## Part 1: Predicting Heights

In [ ]:
original <- read_csv("data/family_heights.csv", show_col_types = FALSE)

heights <- tibble(
  MidParent = original$midparentHeight,
  Child = original$childHeight
)

heights

In [ ]:
heights |>
  ggplot(aes(x = MidParent, y = Child)) +
  geom_point(alpha = 0.5) +
  labs(title = "Child's height and midparent height",
       x = "Midparent height (inches)",
       y = "Child's adult height (inches)") +
  theme_minimal(base_size = 16)

### Nearest Neighbor Regression

In [ ]:
predict_child <- function(mpht) {
  close_points <- heights |>
    filter(MidParent >= mpht - 0.5, MidParent < mpht + 0.5)
  mean(close_points$Child)
}

predict_child(70.48)

In [ ]:
heights_with_predictions <- heights |>
  mutate(Prediction = map_dbl(MidParent, predict_child))

heights_with_predictions

In [ ]:
heights_with_predictions |>
  ggplot(aes(x = MidParent)) +
  geom_point(aes(y = Child), alpha = 0.5) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2.5) +
  labs(title = "Nearest neighbor predictions (gold)",
       x = "Midparent height (inches)",
       y = "Child's adult height (inches)") +
  theme_minimal(base_size = 16)

### What About 74.5 Inches?

In [ ]:
predict_child(74.5)

In [ ]:
heights |>
  filter(MidParent > 73) |>
  count(MidParent)

## Part 2: Where Is the Prediction Line?

In [ ]:
r_table <- function(r) {
  set.seed(42)
  x <- rnorm(1000)
  z <- rnorm(1000)
  tibble(x = x, y = r * x + sqrt(1 - r^2) * z)
}

In [ ]:
su_plot <- function(table) {
  table |>
    ggplot(aes(x = x, y = y)) +
    geom_point(alpha = 0.5) +
    coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
    labs(x = "x (standard units)", y = "y (standard units)") +
    theme_minimal(base_size = 16)
}

In [ ]:
nn_prediction_example <- function(x_val) {
  neighbors <- example |>
    filter(between(x, x_val - 0.25, x_val + 0.25))
  mean(neighbors$y)
}

### r = 0.99

In [ ]:
example <- r_table(0.99)
su_plot(example)

In [ ]:
nn_prediction_example(-2.25)

In [ ]:
example <- example |>
  mutate(Prediction = map_dbl(x, nn_prediction_example))

su_plot(example) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2)

### r = 0

In [ ]:
example <- r_table(0)
su_plot(example)

In [ ]:
example <- example |>
  mutate(Prediction = map_dbl(x, nn_prediction_example))

su_plot(example) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2) +
  geom_abline(slope = 0, intercept = 0, color = "dodgerblue3", linewidth = 1.2)

### r = 0.5

In [ ]:
example <- r_table(0.5)

su_plot(example) +
  geom_vline(xintercept = 1.5, linewidth = 1) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2)

In [ ]:
nn_prediction_example(1.5)

In [ ]:
example <- example |>
  mutate(Prediction = map_dbl(x, nn_prediction_example))

su_plot(example) +
  geom_vline(xintercept = 1.5, linewidth = 1) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2)

In [ ]:
su_plot(example) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2) +
  geom_abline(slope = 0.5, intercept = 0, color = "dodgerblue3", linewidth = 1.2)

### r = 0.7

In [ ]:
example <- r_table(0.7)
example <- example |>
  mutate(Prediction = map_dbl(x, nn_prediction_example))

su_plot(example) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2) +
  geom_abline(slope = 0.7, intercept = 0, color = "dodgerblue3", linewidth = 1.2)

Regression line in standard units: $\text{estimate of } y = r \times x$

## Part 3: The Regression Line in Original Units

$$\text{slope} = r \times \frac{\text{SD of } y}{\text{SD of } x}$$

$$\text{intercept} = \text{average of } y - \text{slope} \times \text{average of } x$$

In [ ]:
family_r <- cor(heights$MidParent, heights$Child)
family_r

In [ ]:
family_slope <- family_r * sd(heights$Child) / sd(heights$MidParent)
family_intercept <- mean(heights$Child) - family_slope * mean(heights$MidParent)
c(family_slope, family_intercept)

### `lm()`

In [ ]:
family_model <- lm(Child ~ MidParent, data = heights)
coef(family_model)

### Predict at 74.5 Inches

In [ ]:
family_slope * 74.5 + family_intercept

In [ ]:
heights_with_predictions <- heights_with_predictions |>
  mutate(`Regression Prediction` = family_slope * MidParent + family_intercept)

heights_with_predictions

In [ ]:
heights_with_predictions |>
  ggplot(aes(x = MidParent)) +
  geom_point(aes(y = Child), alpha = 0.5) +
  geom_point(aes(y = Prediction), color = "goldenrod", size = 2.5) +
  geom_abline(slope = family_slope, intercept = family_intercept, color = "dodgerblue3", linewidth = 1.2) +
  labs(title = "Nearest neighbor (gold) and regression line (blue)",
       x = "Midparent height (inches)",
       y = "Child's adult height (inches)") +
  theme_minimal(base_size = 16)

In [ ]:
heights_with_predictions |>
  summarize(average_gap = mean(abs(Prediction - `Regression Prediction`)))

## Try it out!

Predict `body_mass_g` from `flipper_length_mm`.

1. Find $r$ and the slope in standard units
2. Slope and intercept with the formula, then check with `lm()`
3. Predict for a 220 mm flipper
4. What do the slope and intercept mean?

**Done early?** $r$ within each species

In [ ]:
library(palmerpenguins)

# Keep rows with the two variables we need
model_data <- penguins |>
  filter(!is.na(flipper_length_mm), !is.na(body_mass_g)) |>
  select(species, flipper_length_mm, body_mass_g)

model_data

In [ ]:
ggplot(model_data, aes(x = flipper_length_mm, y = body_mass_g, color = species)) +
  geom_point(alpha = 0.8) +
  labs(
    title = "Flipper Length and Body Mass (Palmer Penguins)",
    x = "Flipper Length (mm)",
    y = "Body Mass (g)",
    caption = "Source: palmerpenguins",
    color = "Species"
  ) +
  theme_minimal(base_size = 18)

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Done early? r within each species

## Extra Practice: Hybrid Cars

Predict `msrp` from `acceleration`, with the same steps. Predict for an acceleration of 15.

In [ ]:
hybrid <- read_csv("data/hybrid.csv", show_col_types = FALSE)
hybrid

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here